# Graph Attention Networks

## Definición

Las redes atencionales de grafos (GATs) adaptan el mecanismo de *self-attention* para operar de forma nativa sobre topologías estructuradas. A diferencia de un Transformer tradicional, donde cada ciudad (nodo) atiende al mapa completo de forma global, una GAT restringe el flujo de información: cada ciudad solo puede comunicarse y calcular puntajes de atención con sus **vecinos directos**, definidos previamente por las aristas del grafo.

La diferencia arquitectónica fundamental frente a un Transformer clásico radica en **cómo** se calcula la importancia entre los nodos. Mientras el Transformer utiliza un mecanismo basado en el producto punto ($Q \cdot K^T$) ideal para medir "similitud", la GAT emplea un **mecanismo de atención aditivo**. La red proyecta el valor de la ciudad de origen y el de la ciudad destino, y los suma a través de un vector de pesos entrenable. Esto permite aprender relaciones direccionales asimétricas (viajar de A hacia B no es lo mismo que de B hacia A) y utilizar la propia conectividad del grafo como un sistema de referencia espacial.

En el contexto del TSP, las GATs encajan a la perfección con la técnica de **esparsificación** geométrica. Al procesar el grafo reducido por la triangulación de Delaunay, inyectamos un poderoso sesgo inductivo: el modelo ignora el ruido de conexiones distantes improbables y concentra su capacidad en dominar la topología de los vecindarios locales. Más aún, esta arquitectura es tan versátil que nos permite inyectar información directamente en las conexiones (como la distancia exacta entre nodos). Esto abre la puerta a resolver variantes del problema donde no existen coordenadas espaciales y solo disponemos de la matriz de distancias, pudiendo aplicarse la GAT directamente sobre el grafo completo.

## Generación de datos

Antes de explicar los componentes arquitectónicos propios de una GAT, generaremos los datos de entrenamiento utilizando la técnica de **esparsificación**. 

In [5]:
from instances.instances import generate_instances

instances = generate_instances(filename="TSP50.pkl", instance_count=1000, cities=50, seed=42)

from data.generation import generate_train_data
from data.adapters.input.sparse import SparseInputAdapter
from data.adapters.output.default import DefaultOutputAdapter

input_config = (SparseInputAdapter, 50)
output_config = (DefaultOutputAdapter, 50)

generate_train_data(
    instance_file="TSP50.pkl", 
    data_filename="train_data.h5", 
    input_adapter_config=input_config, 
    output_adapter_config=output_config,
    size=5000,
    sparse=True
)

from data.preprocessing import load_dataset, split_dataset

train_file, val_file = split_dataset("train_data.h5", 4000)

train_dataset = load_dataset(train_file)
val_dataset = load_dataset(val_file)

Datos guardados en: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5 (Tamaño: 5000)
Cargando dataset original: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5
Guardando Train puro (4000 muestras) en: train_data_train.h5
Guardando Val puro (1000 muestras) en: train_data_val.h5
Dataset train_data_train.h5 cargado con 4000 muestras.
Dataset train_data_val.h5 cargado con 1000 muestras.


## Explicación de la arquitectura

Para comprender cómo el modelo procesa el grafo de ciudades, vamos a desglosar su funcionamiento interno. En términos prácticos, la red atencional de grafos operará simplemente como una capa más dentro de nuestra arquitectura base. Su propósito es **reemplazar al bloque de *Multi-Head Attention* (MHA) tradicional exclusivamente en el *encoder***. 

Es decir, al sustituir únicamente el mecanismo encargado de crear la representación inicial (estructural) del problema, logramos combinar la fórmula matemática pura introducida por Veličković (2018) con las conexiones estabilizadoras de un Transformer moderno, manteniendo intacto el funcionamiento de nuestro *decoder*.

Podemos dividir el *forward pass* de esta capa en 5 etapas lógicas:

### 1. Proyección lineal inicial ($\mathbf{W}$)

Antes de que los nodos interactúen, necesitamos proyectar sus características de entrada (coordenadas o estados previos) a un espacio de mayor dimensionalidad. Al ser un grafo homogéneo, usamos una única matriz de pesos compartida ($\mathbf{W}$) para todos los nodos.

Además, para estabilizar el aprendizaje, dividimos este nuevo vector en múltiples "cabezales" (MHA), permitiendo que la red aprenda diferentes tipos de relaciones topológicas en paralelo.

```python
# Proyectamos y separamos en múltiples cabezales: (B, H, N, head_dim)
Wh = self.W(h).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
```

### 2. Cálculo de pesos de atención (Atención Aditiva)

Aquí ocurre la verdadera "magia" de la GAT. Necesitamos calcular el puntaje direccional $e_{ij}$ entre cada par de nodos. En lugar de concatenar explícitamente los vectores de las ciudades (lo cual consumiría muchísima memoria), utilizamos un truco algebraico: calculamos cómo se ve la ciudad desde la perspectiva de origen (`score_L`), cómo se ve desde el destino (`score_R`), y sumamos ambos valores.

$$e_{ij} = \text{LeakyReLU}\left(a_L^T \mathbf{W}h_i + a_R^T \mathbf{W}h_j\right)$$

```python
score_L = (Wh * self.a_L).sum(dim=-1) # Perspectiva de origen (B, H, N)
score_R = (Wh * self.a_R).sum(dim=-1) # Perspectiva de destino (B, H, N)

# Broadcasting para armar la matriz NxN: e_ij = score_i + score_j
scores = score_L.unsqueeze(-1) + score_R.unsqueeze(-2) # (B, H, N, N)

# Función de activación no lineal
e_ij = self.leakyrelu(scores)
```

### 3. Enmascaramiento y Softmax

El tensor `scores` contiene puntajes para todas las posibles conexiones del grafo ($N \times N$). Para inyectar nuestro sesgo inductivo (la esparsificación de Delaunay o K-NN), utilizamos la matriz de adyacencia como máscara. Reemplazamos los puntajes de las conexiones inexistentes con infinito negativo (`-inf`).

Al aplicar la función Softmax, esos infinitos se convierten en ceros absolutos. El resultado es el tensor $\alpha_{ij}$: una distribución de probabilidad donde cada nodo reparte el 100% de su atención únicamente entre sus vecinos válidos.

```python
# Bloqueamos la atención donde no existe arista en el grafo
_adj_mask = adj_mask.unsqueeze(1).expand_as(e_ij) 
e_ij = e_ij.masked_fill(_adj_mask, float('-inf'))

# Transformamos los puntajes en pesos que sumen 1
alpha = torch.softmax(e_ij, dim=-1)
alpha = self.dropout(alpha)
```

### 4. Suma ponderada (Paso de mensajes)

Con los pesos definidos, cada ciudad absorbe la información de su entorno. Realizamos una multiplicación de matrices entre los pesos de atención $\alpha$ y las características proyectadas de los vecinos $\mathbf{W}h$. Si un vecino resulta ser vital para la ruta, su información dominará este resumen local. Finalmente, concatenamos los múltiples cabezales de vuelta a la dimensión original.

```python
# Suma ponderada de los vecinos: (B, H, N, N) x (B, H, N, head_dim)
h_prime = torch.matmul(alpha, Wh) # (B, H, N, head_dim)

# Concatenación de cabezales (B, N, D)
h_prime = h_prime.transpose(1, 2).contiguous().view(B, N, D)
```

### 5. Actualización (Residual y FFN)

Si solo aplicáramos el paso anterior reiteradas veces, las representaciones de los nodos colapsarían en un promedio borroso (un problema conocido en GNNs como *oversmoothing*). Para evitarlo, tomamos prestada la estructura de actualización del Transformer.

Sumamos el nuevo contexto del vecindario con la identidad original del nodo (conexión residual) y lo normalizamos. Luego, pasamos esta representación por una pequeña red neuronal local (Feed-Forward) para consolidar el aprendizaje no lineal.

```python
# Conexión residual 1 (Preservamos la identidad original)
h = self.norm(h + self.out_proj(h_prime))

# Conexión residual 2 (Feed Forward Network local)
h = self.norm_ffn(h + self.ffn(h))
```

Con esto, el código completo de nuestra capa GAT queda implementado de la siguiente manera:

In [2]:
import torch
import torch.nn as nn

class TSP_GATLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, alpha=0.2, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        
        assert embed_dim % num_heads == 0 
        self.head_dim = embed_dim // num_heads

        # 1. Proyección
        self.W = nn.Linear(embed_dim, embed_dim, bias=False)
        
        # 2. Vectores de atención (separados)
        self.a_L = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        self.a_R = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.xavier_uniform_(self.a_L.data, gain=1.414)
        nn.init.xavier_uniform_(self.a_R.data, gain=1.414)

        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)
        
        # 5. Componentes de actualización
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.norm = nn.LayerNorm(embed_dim)
        
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm_ffn = nn.LayerNorm(embed_dim)

    def forward(self, h, adj_mask):
        B, N, D = h.shape

        # 1. Proyección lineal
        Wh = self.W(h).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. Cálculo de atención aditiva
        score_L = (Wh * self.a_L).sum(dim=-1)
        score_R = (Wh * self.a_R).sum(dim=-1)
        scores = score_L.unsqueeze(-1) + score_R.unsqueeze(-2) 
        e_ij = self.leakyrelu(scores)

        # 3. Enmascaramiento y Softmax
        _adj_mask = adj_mask.unsqueeze(1).expand_as(e_ij) 
        e_ij = e_ij.masked_fill(_adj_mask, float('-inf'))
        alpha = torch.softmax(e_ij, dim=-1)
        alpha = self.dropout(alpha)

        # 4. Agregación (Suma ponderada)
        h_prime = torch.matmul(alpha, Wh)
        h_prime = h_prime.transpose(1, 2).contiguous().view(B, N, D)
        
        # 5. Actualización
        h = self.norm(h + self.out_proj(h_prime))
        h = self.norm_ffn(h + self.ffn(h))
        
        return h

# Incorporando información de las aristas (EGATs)

El modelo GAT que acabamos de definir es sumamente potente para entender topologías, pero presenta una limitante crítica para problemas de optimización geométrica como el TSP: solo sabe **quién** está conectado con quién, pero ignora completamente **a qué distancia** se encuentran. Esperar que el modelo deduzca implícitamente la distancia a partir de las coordenadas $X, Y$ en cada paso es ineficiente.

Para solucionar esto, introducimos las **EGATs (Edge-Augmented Graph Attention Networks)**. Esta variante adapta la capa atencional para que consuma directamente la matriz de distancias (o cualquier matriz de costos). Esta evolución natural del mecanismo aditivo requiere únicamente tres modificaciones matemáticas sobre nuestra capa original.

### 1. Proyección de la Arista (Edge Embedding)

No podemos sumar directamente una distancia escalar (1D) con las características ocultas de un nodo. Necesitamos aplicar una proyección lineal independiente para que la distancia se transforme en un vector de alta dimensionalidad, permitiendo que las aristas "hablen el mismo idioma" que las ciudades.

```python
# --- 1. PROYECCIÓN DE LA ARISTA ---
# En el __init__: Proyectamos el escalar de distancia
self.W_e = nn.Linear(1, embed_dim, bias=False)
self.a_E = nn.Parameter(torch.empty(1, num_heads, 1, 1, self.head_dim))

# En el forward: (B, N, N, 1) -> (B, H, N, N, head_dim)
We = self.W_e(edge_attr).view(B, N, N, self.num_heads, self.head_dim).permute(0, 3, 1, 2, 4)
```

### 2. Modificación del cálculo de Atención

La importancia que la ciudad $i$ le asigna a la ciudad $j$ ya no depende solo de sus respectivas identidades, sino del costo de viajar entre ellas. Al mecanismo aditivo original le agregamos una tercera perspectiva: la de la propia arista multiplicada por su vector de pesos $a_E$.

$$e_{ij} = \text{LeakyReLU}\left(a_L^T \mathbf{W}h_i + a_R^T \mathbf{W}h_j + a_E^T \mathbf{W}_e d_{ij}\right)$$

```python
# --- 2. CÁLCULO DE ATENCIÓN (Con perspectiva de arista) ---
score_L = (Wh * self.a_L).sum(dim=-1) # (B, H, N)
score_R = (Wh * self.a_R).sum(dim=-1) # (B, H, N)

# Calculamos el score de la distancia proyectada
score_E = (We * self.a_E).sum(dim=-1) # (B, H, N, N)

# Sumamos las tres perspectivas
scores = score_L.unsqueeze(-1) + score_R.unsqueeze(-2) + score_E 
e_ij = self.leakyrelu(scores)
```

### 3. Mensajes condicionados por la arista (Agregación)

Durante el paso de mensajes, la ciudad emisora ya no envía únicamente su propia información. Ahora "empaqueta" su identidad junto con la información del camino que la une al receptor. Sumamos matemáticamente el vector del nodo y el vector de la arista **antes** de multiplicar por los pesos de atención $\alpha$.

Debido a que ahora cada conexión tiene un vector único, reemplazamos la multiplicación de matrices matricial (`torch.matmul`) por una suma ponderada explícita.

```python
# --- 3. AGREGACIÓN (Paso de mensajes condicionado) ---
Wh_j = Wh.unsqueeze(2) # (B, H, 1, N, head_dim)

# El mensaje consolida Nodo + Arista
messages = Wh_j + We   # (B, H, N, N, head_dim)

# Suma ponderada explícita aplicando la distribución Softmax (alpha)
h_prime = (alpha.unsqueeze(-1) * messages).sum(dim=3) 
```

# Entrenamiento y validación

A continuación, entrenaremos y validaremos las arquitecturas propuestas:

In [12]:
from models.gat import TSP_GATModel
from models.egat import TSP_EGATModel
from training.sl import sl_train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

# 1. Parámetros compartidos
model_params = {
    "input_dim": 2,
    "embed_dim": 64,
    "num_heads": 4,
    "num_encoder_layers": 2,
    "num_glimpses": 2,
    "dropout_rate": 0.1
}

train_params = {
    "epochs": 20,
    "train_set": train_dataset,
    "val_set": val_dataset,
    "batch_size": 64,
    "lr_config": LRConfig(value=1e-4),
    "loss_fn": CrossEntropyLoss(),
    "metrics": [Accuracy()]
}

# 2. Instanciación y entrenamiento de la red GAT
print("--- Entrenando GAT ---")
gat_model = TSP_GATModel(**model_params)
gat_model = sl_train(
    model=gat_model, 
    metrics_filename="gat_metrics.txt", 
    **train_params
)

# 3. Instanciación y entrenamiento de la red EGAT
print("\n--- Entrenando EGAT ---")
egat_model = TSP_EGATModel(**model_params)
egat_model = sl_train(
    model=egat_model, 
    metrics_filename="egat_metrics.txt", 
    **train_params
)

--- Entrenando GAT ---
** Usando dispositivo: cpu

Epoch 1/20
    Train CrossEntropy: 2.4332 | Val CrossEntropy: 1.6153
    Accuracy: 59.50%
Epoch 2/20
    Train CrossEntropy: 1.4508 | Val CrossEntropy: 1.2552
    Accuracy: 62.80%
Epoch 3/20
    Train CrossEntropy: 1.1692 | Val CrossEntropy: 0.9707
    Accuracy: 72.90%
Epoch 4/20
    Train CrossEntropy: 0.9380 | Val CrossEntropy: 0.8546
    Accuracy: 75.40%
Epoch 5/20
    Train CrossEntropy: 0.8459 | Val CrossEntropy: 0.7835
    Accuracy: 77.30%
Epoch 6/20
    Train CrossEntropy: 0.7823 | Val CrossEntropy: 0.7240
    Accuracy: 79.10%
Epoch 7/20
    Train CrossEntropy: 0.7329 | Val CrossEntropy: 0.6811
    Accuracy: 79.50%
Epoch 8/20
    Train CrossEntropy: 0.6952 | Val CrossEntropy: 0.6631
    Accuracy: 80.40%
Epoch 9/20
    Train CrossEntropy: 0.6669 | Val CrossEntropy: 0.6420
    Accuracy: 81.60%
Epoch 10/20
    Train CrossEntropy: 0.6443 | Val CrossEntropy: 0.6313
    Accuracy: 81.40%
Epoch 11/20
    Train CrossEntropy: 0.6257 | Val

In [13]:
from solvers.eval import evaluate
from data.adapters.input.sparse import SparseInputAdapter

# Configuración compartida
input_config = (SparseInputAdapter, 50)
instance_file = "benchmarks/B50.pkl"

# Agrupamos los modelos en un diccionario para evaluar secuencialmente
models_to_evaluate = {
    "GAT": gat_model,
    "EGAT": egat_model,
}

# Diccionario para almacenar las soluciones sin sobrescribirlas
evaluation_results = {}

for name, model in models_to_evaluate.items():
    print(f"\n{'='*50}")
    print(f"Evaluando: {name}")
    print(f"{'='*50}")
    
    model_sols, ort_sols = evaluate(
        model=model,
        instance_file=instance_file,
        input_adapter_config=input_config,
        num_workers=None,  # Utiliza todos los núcleos de CPU disponibles
        sparse=True
    )
    
    # Guardamos los resultados por si se necesitan analizar o graficar posteriormente
    evaluation_results[name] = {
        "model_sols": model_sols,
        "ort_sols": ort_sols
    }


Evaluando: GAT
Iniciando evaluación conjunta de 100 instancias con 12 workers...

RESULTADOS DE LA VALIDACIÓN
Costo promedio Modelo:   6.34
Costo promedio OR-Tools: 5.82
Gap de optimalidad:      8.87% ± 5.44%


Evaluando: EGAT
Iniciando evaluación conjunta de 100 instancias con 12 workers...

RESULTADOS DE LA VALIDACIÓN
Costo promedio Modelo:   6.34
Costo promedio OR-Tools: 5.82
Gap de optimalidad:      9.01% ± 6.24%

